# 9장 실습 ④ — 진짜 ImageNet 모델

**Keras 3 판**

§9.3에서는 **직접 학습시킨** 원천 과제로 전이학습을 했습니다.
실무에서는 그러지 않습니다. **이미 잘 학습된 것을 받아 씁니다.**

> **가중치를 못 받는 환경이면** 저장소의 `data/pretrained/` 에 파일을
> 넣어 두십시오. 넣는 법은 부록 C에 있습니다.

## 9.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 9.1 목표 과제 — CIFAR-10

§9.3에서 쓴 합성 도형 대신 **진짜 사진**을 씁니다.
32×32 컬러, 10종류.

**학습 데이터를 100 → 500 → 2,000장으로 줄여 가며** 봅니다.
전이학습의 값어치는 **데이터가 적을 때** 드러나기 때문입니다.

In [ ]:
SIZE = 96          # ImageNet 모델은 224로 학습됐지만 96에서도 통합니다

# ★ scale=False — 0~255 그대로 받습니다.
#   preprocess_input 이 그 범위를 기대하기 때문입니다. (§9.4의 함정 ①)
s = data.cifar10(scale=False)
print(s.summary())
print("클래스:", " ".join(data.CIFAR10_CLASSES))

# 시험 2,000장을 **고정**합니다. 학습 데이터만 줄입니다. (7장 §7.6)
rng = np.random.default_rng(42)
ti = rng.choice(len(s.x_test), 2000, replace=False)
X_TE, Y_TE = s.x_test[ti], s.y_test[ti]

def subset(n, seed=42):
    """클래스마다 n/10 장씩 고르게 뽑는다. (층화 — 4장 §4.2)"""
    r = np.random.default_rng(seed)
    idx = np.concatenate([r.permutation(np.where(s.y_train == c)[0])[:n // 10]
                          for c in range(10)])
    return s.x_train[idx], s.y_train[idx]

SIZES = [100, 500] if dlbook.smoke.is_smoke() else [100, 500, 2000]
sets = {n: subset(n) for n in SIZES}
print()
for n in SIZES:
    print(f"  학습 {n:>5}장 (클래스당 {n // 10}장)")
print(f"  시험 {len(X_TE):>5}장 — 고정")

## 9.2 특징 추출 — 여기만 판마다 다릅니다

**얼린 모델의 특징을 한 번만 계산해 두고**, 그 위에 분류부만
학습합니다. 그래서 빠릅니다.

| 판 | 쓰는 모델 |
|---|---|
| Keras 3 / TensorFlow | VGG16, ResNet50 (`keras.applications`) |
| PyTorch | ResNet18 (`torchvision.models`) |

모델이 달라도 **논증은 같습니다.**

In [ ]:
import keras
from keras import layers

PRETRAINED = ["VGG16", "ResNet50"]
COLS = ["처음부터"] + PRETRAINED
MAIN = "ResNet50"

_APP = {"VGG16": (keras.applications.VGG16, keras.applications.vgg16.preprocess_input),
        "ResNet50": (keras.applications.ResNet50, keras.applications.resnet50.preprocess_input)}
_BASE, _FEAT = {}, {}

def _resize(x):
    return np.array(keras.ops.image.resize(x.astype("float32"), (SIZE, SIZE)), copy=True)

def _features(name, x, naive=False):
    """사전학습 모델의 특징을 뽑는다. **얼려 두었으므로 한 번만 계산하면 된다.**"""
    key = (name, naive, x.shape[0], float(x[:3].sum()))
    if key in _FEAT:
        return _FEAT[key]
    Fn, pre = _APP[name]
    if name not in _BASE:
        b = Fn(weights="imagenet", include_top=False,
               input_shape=(SIZE, SIZE, 3), pooling="avg")
        b.trainable = False                      # ← 얼린다 (§9.2)
        _BASE[name] = b
    r = _resize(x)
    inp = (r / 255.0) if naive else pre(r.copy())     # ← 함정 ①
    f = np.asarray(_BASE[name].predict(inp, verbose=0, batch_size=64))
    _FEAT[key] = f
    return f

def pretrained_acc(name, xtr, ytr, xte, yte, naive=False, seed=42, epochs=40):
    """특징 추출 + 분류부만 학습. **이 함수만 판마다 다릅니다.**"""
    ftr, fte = _features(name, xtr, naive), _features(name, xte, naive)
    dlbook.set_seed(seed)
    m = keras.Sequential([layers.Input(shape=(ftr.shape[1],)),
                          layers.Dense(10, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss="sparse_categorical_crossentropy")
    m.fit(ftr, ytr, epochs=dlbook.smoke.epochs(epochs), batch_size=32, verbose=0)
    return metrics.accuracy(yte, m.predict(fte, verbose=0).argmax(1))

def scratch_acc(xtr, ytr, xte, yte, seed=42, epochs=30):
    """비교 대상 — 처음부터 학습하는 작은 CNN."""
    dlbook.set_seed(seed)
    m = keras.Sequential([
        layers.Input(shape=(32, 32, 3)), layers.Rescaling(1 / 255.),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Flatten(), layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss="sparse_categorical_crossentropy")
    m.fit(xtr, ytr, epochs=dlbook.smoke.epochs(epochs), batch_size=32, verbose=0)
    return metrics.accuracy(yte, m.predict(xte, verbose=0).argmax(1))

## 9.3 데이터가 적을수록 격차가 커집니다

In [ ]:
print(f"{'학습 데이터':<12}" + "".join(f"{c:>13}" for c in COLS))
print("-" * (12 + 13 * len(COLS)))

for n in SIZES:
    x, y = sets[n]
    row = [scratch_acc(x, y, X_TE, Y_TE)]
    for name in PRETRAINED:
        row.append(pretrained_acc(name, x, y, X_TE, Y_TE))
    print(f"{n:<12}" + "".join(f"{v:>13.3f}" for v in row))
    for c, v in zip(COLS, row):
        dlbook.record(f"ch09i_n{n}_{c}", v)

print()
print("★ **사전학습 모델은 100장으로, 처음부터 학습한 것이 2,000장으로 낸 것보다")
print("  더 잘합니다.** 20분의 1의 데이터입니다.")
print("→ 이것이 전이학습을 쓰는 이유입니다. 데이터가 적을수록 격차가 큽니다.")

## 9.4 ★ 전처리를 빠뜨리면

§9.4 본문의 「자주 막힙니다」 함정 ①입니다. **숫자로 확인합니다.**

In [ ]:
# ★ §9.4의 함정 ① — 전처리를 안 하면 어떻게 되는가.
# 오류가 나지 않습니다. **성능만 조용히 무너집니다.**
print(f"{'학습 데이터':<12}{'제대로':>12}{'/255만':>12}{'잃은 폭':>12}")
print("-" * 48)
for n in SIZES:
    x, y = sets[n]
    ok = pretrained_acc(MAIN, x, y, X_TE, Y_TE)
    bad = pretrained_acc(MAIN, x, y, X_TE, Y_TE, naive=True)
    print(f"{n:<12}{ok:>12.3f}{bad:>12.3f}{ok - bad:>12.3f}")
    dlbook.record(f"ch09i_pre_n{n}_ok", ok)
    dlbook.record(f"ch09i_pre_n{n}_naive", bad)

print()
print("★ **전처리 한 줄을 빠뜨리면 0.80이 0.28이 됩니다.**")
print("  오류 메시지는 나오지 않습니다. 그냥 성능이 안 나옵니다.")
print("→ 전이학습이 '안 통한다'고 할 때 가장 먼저 볼 곳입니다.")

## 정리

| 학습 데이터 | 처음부터 | VGG16 | ResNet50 |
|:--:|:--:|:--:|:--:|
| 100 | 0.249 | 0.345 | **0.615** |
| 500 | 0.360 | 0.593 | **0.750** |
| 2,000 | 0.455 | 0.718 | **0.801** |

- **사전학습 ResNet50은 100장으로 0.615를 냅니다.** 처음부터 학습한
  모델은 **2,000장으로도 0.455**입니다. **20배의 데이터를 이깁니다.**
- **데이터가 적을수록 격차가 큽니다.** 100장에서 2.5배, 2,000장에서 1.8배.
- VGG16보다 ResNet50이 낫습니다. **잔차 연결**로 훨씬 깊게 쌓은
  모델입니다 (§9.4 본문).

### 전처리

| 학습 데이터 | 제대로 | `/255`만 | 잃은 폭 |
|:--:|:--:|:--:|:--:|
| 2,000 | 0.801 | **0.282** | **0.520** |

**오류가 나지 않습니다. 성능만 조용히 무너집니다.**
전이학습이 "안 통한다"고 할 때 가장 먼저 볼 곳입니다.

### 연습

1. `SIZE` 를 96에서 128, 160으로 키우십시오. 나아집니까.
   시간은 얼마나 늘어납니까.
2. 학습 데이터를 5,000장, 10,000장으로 늘리십시오.
   **어느 지점에서 「처음부터」가 따라잡습니까.**
3. 분류부를 `Dense(10)` 대신 `Dense(128, relu) → Dense(10)` 으로
   바꾸십시오. 나아집니까. **왜 별로 안 나아집니까.**
4. **미세조정**을 해 보십시오 — 마지막 블록만 풀고 학습률 1/10로.
   특징 추출보다 나아집니까. (§9.2의 순서를 지키십시오)
5. **[열린 문제]** ImageNet 정확도가 더 높은 모델(EfficientNet 등)이
   CIFAR-10에서도 더 낫습니까. **본문 §9.4의 주장을 확인하십시오.**